# Walmart Sales Forecasting — XGBoost (v3)

**მოდელი:** XGBoost Regressor
**კატეგორია:** Tree-based (Gradient Boosting)
**MLflow ექსპერიმენტი:** `XGBoost_Training`
**Runs:**
1. `XGBoost_EDA` — exploratory data analysis
2. `XGBoost_Cleaning` — preprocessing pipeline validation
3. `XGBoost_Feature_Selection` — feature set comparison
4. `XGBoost_CrossValidation` — TimeSeriesSplit CV (early-stopped)
5. `XGBoost_HyperparameterTuning` — expanded configuration search (early-stopped)
6. `XGBoost_Final` — საუკეთესო Pipeline (raw → prediction)

**მეტრიკა:** WMAE — competition-ის ოფიციალური. სადღესასწაულო კვირებს ×5 წონა.

**v3 ცვლილებები:** გაფართოებული Store×Dept lag/rolling/momentum ფიჩერების ნაკრები, GPU-ავტოდეტექცია, early-stopping ყველა tuning/CV ეტაპზე (`eval_metric='mae'` + holiday-წონები `sample_weight_eval_set`-ში, რაც პირდაპირ WMAE-ს approximირებს), და გაღრმავებული hyperparameter search.

## 1. Setup

In [ ]:
!pip install xgboost mlflow dagshub --quiet

In [ ]:
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import TimeSeriesSplit

import xgboost as xgb
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

print(f"XGBoost version: {xgb.__version__}")
print(f"MLflow version: {mlflow.__version__}")

try:
    subprocess.check_output(['nvidia-smi'])
    DEVICE = 'cuda'
except Exception:
    DEVICE = 'cpu'
print(f"Device: {DEVICE}")
if DEVICE == 'cpu':
    print("WARNING: no GPU detected. Runtime > Change runtime type > GPU (T4) is strongly recommended "
          "before running Section 9 (HyperparameterTuning) - on CPU the deep-tree search can take hours.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/walmart'
DATA_DIR = f'{PROJECT_DIR}/data'
print(f"Data directory: {DATA_DIR}")

In [ ]:
import dagshub

DAGSHUB_USERNAME = "zberi23"
DAGSHUB_REPO = "walmart-forecasting"

dagshub.init(repo_owner=DAGSHUB_USERNAME, repo_name=DAGSHUB_REPO, mlflow=True)

EXPERIMENT_NAME = "XGBoost_Training"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")

## 2. Raw Data Loading

Pipeline იმუშავებს raw test set-ზე. ამიტომ სამივე original ფაილს ვტვირთავთ, არა preprocessed CSV-ს.

In [ ]:
train_raw = pd.read_csv(f'{DATA_DIR}/train.csv.zip')
test_raw = pd.read_csv(f'{DATA_DIR}/test.csv.zip')
stores_raw = pd.read_csv(f'{DATA_DIR}/stores.csv')
features_raw = pd.read_csv(f'{DATA_DIR}/features.csv.zip')

for df in [train_raw, test_raw, features_raw]:
    df['Date'] = pd.to_datetime(df['Date'])

print(f"Train raw: {train_raw.shape}")
print(f"Test raw:  {test_raw.shape}")
print(f"Stores:    {stores_raw.shape}")
print(f"Features:  {features_raw.shape}")

train_raw.head(3)

## 3. Exploratory Data Analysis

ვაანალიზებთ სამ ფაილს ცალ-ცალკე და ერთად, სანამ preprocessing pipeline-ს ავაშენებთ.

In [ ]:
eda_run = mlflow.start_run(run_name="XGBoost_EDA")

eda = train_raw.merge(stores_raw, on='Store', how='left') \
                .merge(features_raw, on=['Store', 'Date', 'IsHoliday'], how='left')

mlflow.log_param("eda_rows", eda.shape[0])
mlflow.log_param("eda_cols", eda.shape[1])

print("Merged EDA frame:", eda.shape)
eda.describe(include='all').T.head(20)

### 3.1 Missing values

In [ ]:
missing = eda.isnull().mean().sort_values(ascending=False) * 100
missing = missing[missing > 0]

plt.figure(figsize=(10, 5))
plt.barh(missing.index[::-1], missing.values[::-1])
plt.xlabel('% missing')
plt.title('Missing values by column')
plt.tight_layout()
plt.show()

for col, pct in missing.items():
    mlflow.log_metric(f"missing_pct_{col}", pct)

print(missing)

### 3.2 Target distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(eda['Weekly_Sales'], bins=100)
axes[0].set_title('Weekly_Sales distribution')
axes[0].set_xlabel('Weekly_Sales')

axes[1].hist(np.log1p(eda['Weekly_Sales'].clip(lower=0)), bins=100)
axes[1].set_title('log1p(Weekly_Sales) distribution')
axes[1].set_xlabel('log1p(Weekly_Sales)')
plt.tight_layout()
plt.show()

mlflow.log_metric("target_mean", eda['Weekly_Sales'].mean())
mlflow.log_metric("target_median", eda['Weekly_Sales'].median())
mlflow.log_metric("target_std", eda['Weekly_Sales'].std())
mlflow.log_metric("target_negative_count", int((eda['Weekly_Sales'] < 0).sum()))

print(eda['Weekly_Sales'].describe())
print(f"Negative Weekly_Sales rows: {(eda['Weekly_Sales'] < 0).sum()}")

### 3.3 Seasonality and holiday effect

In [ ]:
weekly_avg = eda.groupby('Date')['Weekly_Sales'].mean()

plt.figure(figsize=(14, 5))
plt.plot(weekly_avg.index, weekly_avg.values)
holiday_dates = eda.loc[eda['IsHoliday'], 'Date'].unique()
for d in holiday_dates:
    plt.axvline(pd.Timestamp(d), color='red', alpha=0.15)
plt.title('Average Weekly_Sales over time (red = holiday week)')
plt.xlabel('Date')
plt.ylabel('Avg Weekly_Sales')
plt.tight_layout()
plt.show()

holiday_stats = eda.groupby('IsHoliday')['Weekly_Sales'].agg(['mean', 'median', 'count'])
mlflow.log_metric("holiday_mean_sales", holiday_stats.loc[True, 'mean'])
mlflow.log_metric("non_holiday_mean_sales", holiday_stats.loc[False, 'mean'])

print(holiday_stats)

### 3.4 Store type / size / department effects

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

eda.groupby('Type')['Weekly_Sales'].mean().plot(kind='bar', ax=axes[0], title='Avg sales by store Type')
axes[1].scatter(eda['Size'], eda['Weekly_Sales'], s=2, alpha=0.15)
axes[1].set_title('Size vs Weekly_Sales')
axes[1].set_xlabel('Size')

top_depts = eda.groupby('Dept')['Weekly_Sales'].mean().sort_values(ascending=False).head(15)
top_depts.plot(kind='bar', ax=axes[2], title='Top 15 Dept by avg sales')

plt.tight_layout()
plt.show()

### 3.5 Correlation between numeric covariates

In [ ]:
num_cols = ['Weekly_Sales', 'Size', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
            'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
corr = eda[num_cols].corr()

plt.figure(figsize=(9, 7))
im = plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im)
plt.xticks(range(len(num_cols)), num_cols, rotation=90)
plt.yticks(range(len(num_cols)), num_cols)
plt.title('Correlation matrix')
plt.tight_layout()
plt.show()

mlflow.set_tag("stage", "eda")
mlflow.end_run()

corr['Weekly_Sales'].sort_values(ascending=False)

**დასკვნები EDA-დან:**
- Weekly_Sales ძლიერად skewed არის, Store/Dept/Size ცვლადები ყველაზე მეტ ვარიაციას ხსნიან.
- Holiday კვირები საშუალოდ უფრო მაღალი გაყიდვებით გამოირჩევა.
- Store×Dept ისტორია (lag/rolling) ყველაზე ძლიერი პრედიქტორია, რადგან თითოეული store-department-ის სერია დროში სტაბილურია — ეს არის v3-ის preprocessor-ის მთავარი ფოკუსი.

## 4. Custom Preprocessor

sklearn-compatible Transformer, რომელიც Pipeline-ის შიგნით raw test set-ს (`Store, Dept, Date, IsHoliday`) გარდაქმნის მოდელისთვის მზა ფიჩერების მატრიცად. `fit`-ის დროს, მხოლოდ `train`-ის ისტორიიდან, აგებს:

- **Store×Dept lag ისტორია** — `lag_1..4, lag_8, lag_13, lag_26, lag_52`
- **Store×Dept rolling სტატისტიკა** — `roll_4/8/12/26_mean`, `roll_4/12_std`, `ewm_4`, `ewm_12`
- **Store×Dept expanding სტატისტიკა** — `expanding_mean/median/std`
- **Momentum/ratio ფიჩერები** — `momentum_4_12` (roll_4 − roll_12), `yoy_ratio` (lag_1/lag_52), `recent_vs_hist` (lag_1/expanding_mean)
- **Dept×Week სეზონური საშუალო** და **Store×Dept/Dept median** (hierarchical fallback ახალი კომბინაციებისთვის)

`transform` არასდროს საჭიროებს `Weekly_Sales`-ს — მხოლოდ `fit`-ის დროს დაზოგილ სტატისტიკებს იყენებს, ისე რომ raw `test.csv`-ზეც უცვლელად იმუშაოს.

In [ ]:
class WalmartPreprocessor(BaseEstimator, TransformerMixin):
    """
    Custom Pipeline transformer: raw Walmart CSV-ები -> feature matrix.
    Store x Dept lag/rolling/momentum + Dept x Week seasonal averages,
    fit-il mxolod train-is istoriaze.
    """

    LAG_STEPS = [1, 2, 3, 4, 8, 13, 26, 52]
    ROLL_WINDOWS = [4, 8, 12, 26]
    ROLL_STD_WINDOWS = [4, 12]
    EWM_SPANS = [4, 12]

    def __init__(self, stores_df, features_df):
        self.stores_df = stores_df.copy()
        self.features_df = features_df.copy()

        if not pd.api.types.is_datetime64_any_dtype(self.features_df['Date']):
            self.features_df['Date'] = pd.to_datetime(self.features_df['Date'])

        self._dept_rank_map = None

    def fit(self, X, y=None):
        dept_counts = X['Dept'].value_counts()
        self._dept_rank_map = {d: r for r, d in enumerate(dept_counts.index)}

        if y is None and 'Weekly_Sales' in X.columns:
            y = X['Weekly_Sales'].values

        hist = X[['Store', 'Dept', 'Date']].copy()
        if not pd.api.types.is_datetime64_any_dtype(hist['Date']):
            hist['Date'] = pd.to_datetime(hist['Date'])
        hist['Weekly_Sales'] = np.asarray(y, dtype=float)
        hist = hist.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

        grp = hist.groupby(['Store', 'Dept'])['Weekly_Sales']

        for lag in self.LAG_STEPS:
            hist[f'lag_{lag}'] = grp.shift(lag)

        for w in self.ROLL_WINDOWS:
            hist[f'roll_{w}_mean'] = grp.transform(lambda s, w=w: s.shift(1).rolling(w, min_periods=1).mean())
        for w in self.ROLL_STD_WINDOWS:
            hist[f'roll_{w}_std'] = grp.transform(lambda s, w=w: s.shift(1).rolling(w, min_periods=2).std())
        for span in self.EWM_SPANS:
            hist[f'ewm_{span}'] = grp.transform(lambda s, span=span: s.shift(1).ewm(span=span, min_periods=1).mean())

        hist['store_dept_expanding_mean'] = grp.transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
        hist['store_dept_expanding_median'] = grp.transform(lambda s: s.shift(1).expanding(min_periods=1).median())
        hist['store_dept_expanding_std'] = grp.transform(lambda s: s.shift(1).expanding(min_periods=2).std())

        hist['momentum_4_12'] = hist['roll_4_mean'] - hist['roll_12_mean']
        hist['yoy_ratio'] = hist['lag_1'] / hist['lag_52'].replace(0, np.nan)
        hist['recent_vs_hist'] = hist['lag_1'] / hist['store_dept_expanding_mean'].replace(0, np.nan)

        dept_grp = hist.groupby('Dept')['Weekly_Sales']
        hist['dept_expanding_mean'] = dept_grp.transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
        store_grp = hist.groupby('Store')['Weekly_Sales']
        hist['store_expanding_mean'] = store_grp.transform(lambda s: s.shift(1).expanding(min_periods=1).mean())

        lag_feature_cols = (
            [f'lag_{lag}' for lag in self.LAG_STEPS]
            + [f'roll_{w}_mean' for w in self.ROLL_WINDOWS]
            + [f'roll_{w}_std' for w in self.ROLL_STD_WINDOWS]
            + [f'ewm_{span}' for span in self.EWM_SPANS]
            + ['store_dept_expanding_mean', 'store_dept_expanding_median', 'store_dept_expanding_std',
               'momentum_4_12', 'yoy_ratio', 'recent_vs_hist',
               'dept_expanding_mean', 'store_expanding_mean']
        )
        missing_cols = [c for c in lag_feature_cols if c not in hist.columns]
        assert not missing_cols, f"WalmartPreprocessor.fit did not create expected columns: {missing_cols}"

        global_median = hist['Weekly_Sales'].median()
        ratio_like = ('ratio', 'recent_vs_hist', 'momentum', 'std')
        self._global_fallback = {
            col: (0.0 if any(tag in col for tag in ratio_like) else global_median)
            for col in lag_feature_cols
        }
        self._lag_feature_cols = lag_feature_cols

        self._history_features = hist[['Store', 'Dept', 'Date'] + lag_feature_cols] \
            .sort_values('Date').reset_index(drop=True)

        dw = hist.copy()
        dw['Week'] = dw['Date'].dt.isocalendar().week.astype(int)
        self._dept_week_table = dw.groupby(['Dept', 'Week'])['Weekly_Sales'].mean()
        self._dept_week_global_mean = dw['Weekly_Sales'].mean()

        self._store_dept_median = hist.groupby(['Store', 'Dept'])['Weekly_Sales'].median()
        self._dept_median = hist.groupby('Dept')['Weekly_Sales'].median()
        self._global_median = global_median

        return self

    def transform(self, X):
        df = X.copy()

        if not pd.api.types.is_datetime64_any_dtype(df['Date']):
            df['Date'] = pd.to_datetime(df['Date'])

        df = df.merge(self.features_df, on=['Store', 'Date', 'IsHoliday'], how='left')
        df = df.merge(self.stores_df, on='Store', how='left')

        for col in ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']:
            df[col] = df[col].fillna(0)

        for col in ['CPI', 'Unemployment', 'Temperature', 'Fuel_Price']:
            df[col] = df.groupby('Store')[col].transform(lambda x: x.ffill().bfill())

        df['Year'] = df['Date'].dt.year
        df['Month'] = df['Date'].dt.month
        df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
        df['Day_of_Year'] = df['Date'].dt.dayofyear
        df['Quarter'] = df['Date'].dt.quarter
        df['WeekOfMonth'] = ((df['Date'].dt.day - 1) // 7 + 1).astype(int)

        super_bowl = pd.to_datetime(['2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08'])
        labor_day = pd.to_datetime(['2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06'])
        thanksgiving = pd.to_datetime(['2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29'])
        christmas = pd.to_datetime(['2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27'])
        all_holidays = pd.DatetimeIndex(super_bowl).append(labor_day).append(thanksgiving).append(christmas)

        df['Is_SuperBowl'] = df['Date'].isin(super_bowl).astype(int)
        df['Is_LaborDay'] = df['Date'].isin(labor_day).astype(int)
        df['Is_Thanksgiving'] = df['Date'].isin(thanksgiving).astype(int)
        df['Is_Christmas'] = df['Date'].isin(christmas).astype(int)

        holiday_ns = all_holidays.values.astype('datetime64[D]').astype(np.int64)
        date_ns = df['Date'].values.astype('datetime64[D]').astype(np.int64)
        df['Days_To_Holiday'] = np.abs(date_ns[:, None] - holiday_ns[None, :]).min(axis=1)

        df['Month_Sin'] = np.sin(2 * np.pi * df['Month'] / 12)
        df['Month_Cos'] = np.cos(2 * np.pi * df['Month'] / 12)
        df['Week_Sin'] = np.sin(2 * np.pi * df['Week'] / 52)
        df['Week_Cos'] = np.cos(2 * np.pi * df['Week'] / 52)

        markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
        df['MarkDown_Total'] = df[markdown_cols].sum(axis=1)
        df['MarkDown_Active_Count'] = (df[markdown_cols] > 0).sum(axis=1)
        df['Size_Per_Dept_Rank'] = df['Dept'].map(self._dept_rank_map).fillna(-1)

        size_bins = [0, 50000, 100000, 150000, 200000, np.inf]
        df['Size_Bucket'] = pd.cut(df['Size'], bins=size_bins, labels=False)

        type_mapping = {'A': 0, 'B': 1, 'C': 2}
        df['Type_Encoded'] = df['Type'].map(type_mapping)
        df['IsHoliday'] = df['IsHoliday'].astype(int)

        dw_index = pd.MultiIndex.from_arrays([df['Dept'], df['Week']])
        df['Dept_Week_Seasonal_Avg'] = dw_index.map(self._dept_week_table).to_numpy()
        df['Dept_Week_Seasonal_Avg'] = pd.Series(df['Dept_Week_Seasonal_Avg']).fillna(self._dept_week_global_mean).values

        sd_index = pd.MultiIndex.from_arrays([df['Store'], df['Dept']])
        df['Store_Dept_Median'] = sd_index.map(self._store_dept_median).to_numpy()
        df['Store_Dept_Median'] = pd.Series(df['Store_Dept_Median']).fillna(
            df['Dept'].map(self._dept_median)).fillna(self._global_median).values

        left = df[['Store', 'Dept', 'Date']].sort_values('Date')
        merged_lags = pd.merge_asof(
            left,
            self._history_features,
            on='Date',
            by=['Store', 'Dept'],
            direction='backward'
        ).set_index(left.index).sort_index()
        for col in self._lag_feature_cols:
            df[col] = merged_lags[col].fillna(self._global_fallback[col])

        df = df.drop(columns=['Date', 'Type'])
        return df


preprocessor = WalmartPreprocessor(stores_raw, features_raw)
X_test_transformed = preprocessor.fit(train_raw).transform(test_raw)
print(f"Test raw shape: {test_raw.shape}")
print(f"Test transformed: {X_test_transformed.shape}")
print(f"Columns: {list(X_test_transformed.columns)}")

## 5. WMAE Metric + მონაცემების მომზადება

Train/val split დროზეა დაფუძნებული (ბოლო 10%). Holiday-წონები (`w_train`, `w_val`) გამოიყენება ტრენინგში (`sample_weight`) და შეფასებაში, ისევე როგორც early-stopping-ის `sample_weight_eval_set`-ში — xgboost-ის `eval_metric='mae'` წონიანი ვერსია ზუსტად WMAE-ს ემთხვევა, ასე რომ early stopping უშუალოდ ამ მეტრიკას ეყრდნობა.

In [ ]:
def wmae(y_true, y_pred, weights):
    """Weighted Mean Absolute Error — competition metric."""
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)


def get_weights(is_holiday):
    """Holiday kviras 5, sxvas 1."""
    return np.where(is_holiday == 1, 5, 1)


train_sorted = train_raw.sort_values('Date').reset_index(drop=True)
split_date = train_sorted['Date'].quantile(0.9)

X_train_raw = train_sorted[train_sorted['Date'] < split_date].drop(columns=['Weekly_Sales'])
y_train = train_sorted[train_sorted['Date'] < split_date]['Weekly_Sales'].values

X_val_raw = train_sorted[train_sorted['Date'] >= split_date].drop(columns=['Weekly_Sales'])
y_val = train_sorted[train_sorted['Date'] >= split_date]['Weekly_Sales'].values

w_train = get_weights(X_train_raw['IsHoliday'].values.astype(int))
w_val = get_weights(X_val_raw['IsHoliday'].values.astype(int))

print(f"Train: {X_train_raw.shape}, val: {X_val_raw.shape}")
print(f"Train date range: {X_train_raw['Date'].min()} -> {X_train_raw['Date'].max()}")
print(f"Val date range:   {X_val_raw['Date'].min()} -> {X_val_raw['Date'].max()}")

## 6. Run 1 — `XGBoost_Cleaning`

Preprocessing pipeline validation.

In [ ]:
with mlflow.start_run(run_name="XGBoost_Cleaning"):
    preprocessor = WalmartPreprocessor(stores_raw, features_raw)
    preprocessor.fit(X_train_raw, y_train)

    X_train_processed = preprocessor.transform(X_train_raw)
    X_val_processed = preprocessor.transform(X_val_raw)
    X_test_processed = preprocessor.transform(test_raw)

    train_nans = X_train_processed.isnull().sum().sum()
    val_nans = X_val_processed.isnull().sum().sum()
    test_nans = X_test_processed.isnull().sum().sum()

    mlflow.log_param("train_rows_raw", X_train_raw.shape[0])
    mlflow.log_param("train_rows_processed", X_train_processed.shape[0])
    mlflow.log_param("test_rows_raw", test_raw.shape[0])
    mlflow.log_param("test_rows_processed", X_test_processed.shape[0])
    mlflow.log_param("n_features_after_preprocessing", X_train_processed.shape[1])
    mlflow.log_param("markdown_fill_value", 0)
    mlflow.log_param("cpi_unemployment_temp_fuel_fill_method", "groupby_store_ffill_bfill")
    mlflow.log_param("features", list(X_train_processed.columns))

    mlflow.log_metric("train_missing_after_clean", train_nans)
    mlflow.log_metric("val_missing_after_clean", val_nans)
    mlflow.log_metric("test_missing_after_clean", test_nans)

    mlflow.set_tag("stage", "cleaning")

    print(f"Train NaN after clean: {train_nans}")
    print(f"Val NaN after clean:   {val_nans}")
    print(f"Test NaN after clean:  {test_nans}")
    print(f"Features: {X_train_processed.shape[1]}")

## 7. Run 2 — `XGBoost_Feature_Selection`

- **all** — ყველა ფიჩერი
- **no_markdown** — MarkDown-ების გარეშე
- **core** — მხოლოდ ძირითადი date + store ფიჩერები
- **extended** — core + Store×Dept lag/rolling/momentum ისტორია + Dept×Week/Store×Dept სეზონური სტატისტიკა (v3-ის სრული feature set)

In [ ]:
FEATURE_SETS = {
    "all": list(X_train_processed.columns),

    "no_markdown": [c for c in X_train_processed.columns
                    if c not in ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']],

    "core": ['Store', 'Dept', 'IsHoliday', 'Size', 'Type_Encoded',
             'Year', 'Month', 'Week', 'Day_of_Year',
             'Is_SuperBowl', 'Is_LaborDay', 'Is_Thanksgiving', 'Is_Christmas',
             'Month_Sin', 'Month_Cos', 'Week_Sin', 'Week_Cos'],

    "extended": ['Store', 'Dept', 'IsHoliday', 'Size', 'Type_Encoded',
                 'Year', 'Month', 'Week', 'Day_of_Year', 'Quarter',
                 'WeekOfMonth', 'Is_SuperBowl', 'Is_LaborDay', 'Is_Thanksgiving', 'Is_Christmas',
                 'Days_To_Holiday', 'Month_Sin', 'Month_Cos', 'Week_Sin', 'Week_Cos',
                 'MarkDown_Total', 'MarkDown_Active_Count', 'Size_Bucket', 'Size_Per_Dept_Rank',
                 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
                 'Dept_Week_Seasonal_Avg', 'Store_Dept_Median',
                 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_8', 'lag_13', 'lag_26', 'lag_52',
                 'roll_4_mean', 'roll_8_mean', 'roll_12_mean', 'roll_26_mean',
                 'roll_4_std', 'roll_12_std', 'ewm_4', 'ewm_12',
                 'store_dept_expanding_mean', 'store_dept_expanding_median', 'store_dept_expanding_std',
                 'momentum_4_12', 'yoy_ratio', 'recent_vs_hist',
                 'dept_expanding_mean', 'store_expanding_mean'],
}

BASELINE_PARAMS = {
    'n_estimators': 400,
    'max_depth': 8,
    'learning_rate': 0.1,
    'random_state': 42,
    'tree_method': 'hist',
    'device': DEVICE,
    'n_jobs': -1,
}

feature_selection_results = {}

with mlflow.start_run(run_name="XGBoost_Feature_Selection"):
    mlflow.log_params(BASELINE_PARAMS)
    mlflow.log_param("feature_sets_tested", list(FEATURE_SETS.keys()))
    mlflow.log_param("sample_weighted_training", True)

    for fs_name, feats in FEATURE_SETS.items():
        with mlflow.start_run(run_name=f"FS_{fs_name}", nested=True):
            X_tr = X_train_processed[feats]
            X_va = X_val_processed[feats]

            model = xgb.XGBRegressor(**BASELINE_PARAMS)
            model.fit(X_tr, y_train, sample_weight=w_train, verbose=False)

            preds = model.predict(X_va)
            score = wmae(y_val, preds, w_val)

            feature_selection_results[fs_name] = score

            mlflow.log_param("feature_set", fs_name)
            mlflow.log_param("n_features", len(feats))
            mlflow.log_metric("val_wmae", score)

            print(f"[{fs_name:12}] n_features={len(feats):3d}, WMAE = {score:.2f}")

    best_fs = min(feature_selection_results, key=feature_selection_results.get)
    mlflow.log_metric("best_val_wmae", feature_selection_results[best_fs])
    mlflow.log_param("best_feature_set", best_fs)
    mlflow.set_tag("stage", "feature_selection")

    print(f"\nBest feature set: {best_fs} (WMAE={feature_selection_results[best_fs]:.2f})")

BEST_FEATURES = FEATURE_SETS[best_fs]

## 8. Run 3 — `XGBoost_CrossValidation`

**TimeSeriesSplit** — თითოეულ fold-ში `WalmartPreprocessor` თავიდან fit-დება მხოლოდ იმ fold-ის training slice-ზე. თითოეული fold early-stopping-ით ტრენინგდება (`eval_metric='mae'` + holiday-წონები), რაც overfitting-ს bounded n_estimators-ის მოცდის გარეშე აკონტროლებს.

In [ ]:
N_SPLITS = 5
EARLY_STOP_ROUNDS = 100
MAX_ESTIMATORS_CV = 4000

tscv = TimeSeriesSplit(n_splits=N_SPLITS)
cv_scores = []
cv_best_iters = []

CV_PARAMS = {
    "max_depth": 10,
    "learning_rate": 0.03,
    "subsample": 0.85,
    "colsample_bytree": 0.8,
    "min_child_weight": 3,
    "reg_alpha": 0.05,
    "reg_lambda": 1.2,
    "gamma": 0.0,
    "random_state": 42,
    "tree_method": "hist",
    "device": DEVICE,
    "n_jobs": -1,
}

with mlflow.start_run(run_name="XGBoost_CrossValidation"):
    mlflow.log_params(CV_PARAMS)
    mlflow.log_param("cv_method", "TimeSeriesSplit")
    mlflow.log_param("n_splits", N_SPLITS)
    mlflow.log_param("feature_set", best_fs)
    mlflow.log_param("sample_weighted_training", True)
    mlflow.log_param("early_stopping_rounds", EARLY_STOP_ROUNDS)

    for fold, (tr_idx, va_idx) in enumerate(tscv.split(X_train_raw)):
        X_tr_raw_fold = X_train_raw.iloc[tr_idx]
        y_tr_fold = y_train[tr_idx]
        X_va_raw_fold = X_train_raw.iloc[va_idx]
        y_va_fold = y_train[va_idx]

        w_tr_fold = get_weights(X_tr_raw_fold['IsHoliday'].values.astype(int))
        w_va_fold = get_weights(X_va_raw_fold['IsHoliday'].values.astype(int))

        fold_preprocessor = WalmartPreprocessor(stores_raw, features_raw)
        fold_preprocessor.fit(X_tr_raw_fold, y_tr_fold)

        X_tr_fold = fold_preprocessor.transform(X_tr_raw_fold)[BEST_FEATURES]
        X_va_fold = fold_preprocessor.transform(X_va_raw_fold)[BEST_FEATURES]

        model = xgb.XGBRegressor(
            **CV_PARAMS,
            n_estimators=MAX_ESTIMATORS_CV,
            eval_metric='mae',
            early_stopping_rounds=EARLY_STOP_ROUNDS,
        )
        model.fit(
            X_tr_fold, y_tr_fold, sample_weight=w_tr_fold,
            eval_set=[(X_va_fold, y_va_fold)], sample_weight_eval_set=[w_va_fold],
            verbose=False,
        )

        preds = model.predict(X_va_fold)
        score = wmae(y_va_fold, preds, w_va_fold)

        cv_scores.append(score)
        cv_best_iters.append(model.best_iteration)
        mlflow.log_metric("fold_wmae", score, step=fold)
        mlflow.log_metric("fold_best_iteration", model.best_iteration, step=fold)
        print(f"Fold {fold+1}: WMAE = {score:.2f}, best_iteration = {model.best_iteration}")

    mean_score = np.mean(cv_scores)
    std_score = np.std(cv_scores)

    mlflow.log_metric("cv_wmae_mean", mean_score)
    mlflow.log_metric("cv_wmae_std", std_score)
    mlflow.set_tag("stage", "cross_validation")

    print(f"\nCV WMAE: {mean_score:.2f} +/- {std_score:.2f}")

## 9. Run 4 — `XGBoost_HyperparameterTuning`

გაფართოებული სიღრმის/რეგულარიზაციის საძიებო სივრცე — `n_estimators` ყველგან მაღალ upper bound-ზეა დაყენებული (`MAX_ESTIMATORS`) და ფაქტობრივი boosting round-ების რაოდენობას early stopping ირჩევს (`eval_metric='mae'`, holiday-წონები `sample_weight_eval_set`-ში — ეს ზუსტად WMAE-ს უტოლდება). ეს საშუალებას გვაძლევს გავცადოთ საკმაოდ ღრმა (`max_depth` 10–20, ასევე შეუზღუდავი depth + `grow_policy='lossguide'` + `max_leaves`) კონფიგურაციები overfitting-ის რისკის გარეშე, რადგან თითოეული candidate თავისივე ოპტიმალურ round-ზე ჩერდება.

In [ ]:
MAX_ESTIMATORS = 4000
EARLY_STOP_ROUNDS_TUNE = 75

CANDIDATE_CONFIGS = [
    {"max_depth": 8,  "learning_rate": 0.05, "subsample": 0.80, "colsample_bytree": 0.80, "min_child_weight": 4, "reg_alpha": 0.05, "reg_lambda": 1.2, "gamma": 0.0},
    {"max_depth": 10, "learning_rate": 0.05, "subsample": 0.85, "colsample_bytree": 0.80, "min_child_weight": 4, "reg_alpha": 0.05, "reg_lambda": 1.2, "gamma": 0.0},
    {"max_depth": 10, "learning_rate": 0.03, "subsample": 0.85, "colsample_bytree": 0.80, "min_child_weight": 4, "reg_alpha": 0.05, "reg_lambda": 1.2, "gamma": 0.0},
    {"max_depth": 12, "learning_rate": 0.03, "subsample": 0.85, "colsample_bytree": 0.75, "min_child_weight": 4, "reg_alpha": 0.10, "reg_lambda": 1.5, "gamma": 0.05},
    {"max_depth": 12, "learning_rate": 0.02, "subsample": 0.90, "colsample_bytree": 0.75, "min_child_weight": 3, "reg_alpha": 0.10, "reg_lambda": 1.5, "gamma": 0.05},
    {"max_depth": 14, "learning_rate": 0.02, "subsample": 0.90, "colsample_bytree": 0.70, "min_child_weight": 3, "reg_alpha": 0.10, "reg_lambda": 1.5, "gamma": 0.05},
    {"max_depth": 14, "learning_rate": 0.015, "subsample": 0.90, "colsample_bytree": 0.70, "min_child_weight": 2, "reg_alpha": 0.15, "reg_lambda": 2.0, "gamma": 0.1},
    {"max_depth": 16, "learning_rate": 0.015, "subsample": 0.90, "colsample_bytree": 0.65, "min_child_weight": 2, "reg_alpha": 0.15, "reg_lambda": 2.0, "gamma": 0.1},
    {"max_depth": 16, "learning_rate": 0.01,  "subsample": 0.95, "colsample_bytree": 0.65, "min_child_weight": 2, "reg_alpha": 0.15, "reg_lambda": 2.0, "gamma": 0.1},
    {"max_depth": 18, "learning_rate": 0.01,  "subsample": 0.95, "colsample_bytree": 0.60, "min_child_weight": 1, "reg_alpha": 0.15, "reg_lambda": 2.0, "gamma": 0.1},
    {"max_depth": 20, "learning_rate": 0.008, "subsample": 0.95, "colsample_bytree": 0.60, "min_child_weight": 1, "reg_alpha": 0.20, "reg_lambda": 2.5, "gamma": 0.1},
    {"max_depth": 0,  "grow_policy": "lossguide", "max_leaves": 512,  "learning_rate": 0.02,  "subsample": 0.90, "colsample_bytree": 0.70, "min_child_weight": 2, "reg_alpha": 0.10, "reg_lambda": 1.5, "gamma": 0.05},
    {"max_depth": 0,  "grow_policy": "lossguide", "max_leaves": 1024, "learning_rate": 0.015, "subsample": 0.90, "colsample_bytree": 0.65, "min_child_weight": 1, "reg_alpha": 0.15, "reg_lambda": 2.0, "gamma": 0.1},
]

FIXED_PARAMS = {
    "random_state": 42,
    "tree_method": "hist",
    "device": DEVICE,
    "n_jobs": -1,
}

print(f"Total configurations: {len(CANDIDATE_CONFIGS)}")

tuning_results = []

X_tr_tune = X_train_processed[BEST_FEATURES]
X_va_tune = X_val_processed[BEST_FEATURES]

with mlflow.start_run(run_name="XGBoost_HyperparameterTuning"):
    mlflow.log_param("n_configs", len(CANDIDATE_CONFIGS))
    mlflow.log_param("feature_set", best_fs)
    mlflow.log_param("sample_weighted_training", True)
    mlflow.log_param("max_estimators", MAX_ESTIMATORS)
    mlflow.log_param("early_stopping_rounds", EARLY_STOP_ROUNDS_TUNE)

    for i, combo in enumerate(CANDIDATE_CONFIGS):
        params = {**FIXED_PARAMS, **combo}

        with mlflow.start_run(run_name=f"Config_{i+1}", nested=True):
            model = xgb.XGBRegressor(
                **params,
                n_estimators=MAX_ESTIMATORS,
                eval_metric='mae',
                early_stopping_rounds=EARLY_STOP_ROUNDS_TUNE,
            )
            model.fit(
                X_tr_tune, y_train, sample_weight=w_train,
                eval_set=[(X_va_tune, y_val)], sample_weight_eval_set=[w_val],
                verbose=False,
            )

            preds = model.predict(X_va_tune)
            score = wmae(y_val, preds, w_val)
            best_iter = model.best_iteration

            tuning_results.append({**combo, "n_estimators": best_iter, "val_wmae": score})

            mlflow.log_params(params)
            mlflow.log_param("n_estimators_capped", MAX_ESTIMATORS)
            mlflow.log_metric("best_iteration", best_iter)
            mlflow.log_metric("val_wmae", score)
            print(f"[{i+1}/{len(CANDIDATE_CONFIGS)}] {combo} -> WMAE = {score:.2f} (n_estimators={best_iter})")

    best_combo = min(tuning_results, key=lambda r: r["val_wmae"])
    BEST_PARAMS = {**FIXED_PARAMS, **{k: v for k, v in best_combo.items() if k not in ("val_wmae",)}}

    mlflow.log_metric("best_val_wmae", best_combo["val_wmae"])
    mlflow.log_params(BEST_PARAMS)
    mlflow.set_tag("stage", "hyperparameter_tuning")

    print(f"\nBest params: {BEST_PARAMS}")
    print(f"Best WMAE:   {best_combo['val_wmae']:.2f}")

## 10. Run 5 — `XGBoost_Final` (Pipeline)

საბოლოო sklearn Pipeline:

```
raw test.csv (Store, Dept, Date, IsHoliday)
         ↓
WalmartPreprocessor  (merge + fillna + lag/rolling/momentum + სეზონური FE)
         ↓
FeatureSelector      (best feature subset)
         ↓
XGBoost              (best hyperparameters — n_estimators უკვე early-stopping-ით შერჩეული, holiday sample weight)
         ↓
predictions
```

პირველ რიგში ვზომავთ generalization-ს train-ზე გაწვრთნილი პაიპლაინით hold-out ვალიდაციაზე (`val_wmae`). ამის შემდეგ ცალკე ვაწვრთნით საბოლოო production pipeline-ს მთელ მონაცემზე (train+val) deployment-ისთვის (`full_data_train_wmae`).

In [ ]:
class FeatureSelector(BaseEstimator, TransformerMixin):
    """Pipeline-compatible feature subset selector."""
    def __init__(self, features):
        self.features = features
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X[self.features]


eval_pipeline = Pipeline([
    ('preprocess', WalmartPreprocessor(stores_raw, features_raw)),
    ('select', FeatureSelector(BEST_FEATURES)),
    ('model', xgb.XGBRegressor(**BEST_PARAMS)),
])

X_full_raw = train_sorted.drop(columns=['Weekly_Sales'])
y_full = train_sorted['Weekly_Sales'].values
w_full = get_weights(X_full_raw['IsHoliday'].values.astype(int))

print("Pipeline steps:")
for name, step in eval_pipeline.steps:
    print(f"  {name}: {type(step).__name__}")

In [ ]:
with mlflow.start_run(run_name="XGBoost_Final") as final_run:
    eval_pipeline.fit(X_train_raw, y_train, model__sample_weight=w_train)
    val_preds = eval_pipeline.predict(X_val_raw)
    val_wmae = wmae(y_val, val_preds, w_val)

    final_pipeline = Pipeline([
        ('preprocess', WalmartPreprocessor(stores_raw, features_raw)),
        ('select', FeatureSelector(BEST_FEATURES)),
        ('model', xgb.XGBRegressor(**BEST_PARAMS)),
    ])
    final_pipeline.fit(X_full_raw, y_full, model__sample_weight=w_full)

    full_fit_preds = final_pipeline.predict(X_full_raw)
    full_data_train_wmae = wmae(y_full, full_fit_preds, w_full)

    mlflow.log_params(BEST_PARAMS)
    mlflow.log_param("pipeline_steps", str([s[0] for s in final_pipeline.steps]))
    mlflow.log_param("feature_set", best_fs)
    mlflow.log_param("n_features_used", len(BEST_FEATURES))
    mlflow.log_param("training_rows", len(X_full_raw))
    mlflow.log_param("sample_weighted_training", True)

    mlflow.log_metric("val_wmae", val_wmae)
    mlflow.log_metric("full_data_train_wmae", full_data_train_wmae)

    signature = infer_signature(X_val_raw.head(100), val_preds[:100])

    mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        artifact_path="xgboost_pipeline",
        signature=signature,
        input_example=X_val_raw.head(5),
        registered_model_name="walmart_xgboost",
        serialization_format="cloudpickle"
    )

    mlflow.set_tag("stage", "final")
    mlflow.set_tag("model_type", "XGBoost")

    print(f"\nFinal Pipeline trained")
    print(f"Hold-out val WMAE:        {val_wmae:.2f}")
    print(f"Full-data train WMAE:     {full_data_train_wmae:.2f}")
    print(f"Registered as 'walmart_xgboost' in Model Registry")
    print(f"Run ID: {final_run.info.run_id}")

## 11. Pipeline-ის ვალიდაცია raw test-ზე

დავრწმუნდეთ, რომ Pipeline მუშაობს უცვლელ, preprocessed არა test set-ზე.

In [ ]:
test_predictions = final_pipeline.predict(test_raw)

print(f"Raw test shape: {test_raw.shape}")
print(f"Predictions shape: {test_predictions.shape}")
print(f"\nFirst 5 predictions:")
for i, p in enumerate(test_predictions[:5]):
    print(f"  Row {i}: Store={test_raw.iloc[i]['Store']}, "
          f"Dept={test_raw.iloc[i]['Dept']}, "
          f"Date={test_raw.iloc[i]['Date']}, "
          f"Predicted Weekly_Sales = {p:.2f}")

import joblib
import os
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
joblib.dump(final_pipeline, f'{PROJECT_DIR}/models/xgboost_pipeline.pkl')
print(f"\nPipeline shenaxulia: {PROJECT_DIR}/models/xgboost_pipeline.pkl")

## 12. Feature Importance

In [ ]:
xgb_model = final_pipeline.named_steps['model']

importances = pd.DataFrame({
    'feature': BEST_FEATURES,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 8))
plt.barh(importances['feature'].head(20)[::-1], importances['importance'].head(20)[::-1])
plt.title('XGBoost Top 20 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print("\nTop 15 features:")
print(importances.head(15).to_string(index=False))

## 13. შეჯამება

v3-ის მთავარი ცვლილებები v2-სთან შედარებით:

- **გაფართოებული Store×Dept ისტორია** — lag 1/2/3/4/8/13/26/52, rolling mean (4/8/12/26) და std (4/12), EWM (span 4/12), expanding mean/median/std, momentum (`roll_4 − roll_12`) და ratio (`yoy_ratio`, `recent_vs_hist`) ფიჩერები.
- **Hierarchical fallback** ახალი/იშვიათი Store×Dept კომბინაციებისთვის — `Store_Dept_Median` → `Dept` median → global median.
- **Early stopping ყველა ეტაპზე** (`CrossValidation`, `HyperparameterTuning`) — `eval_metric='mae'` + holiday `sample_weight_eval_set`-ით, რაც პირდაპირ WMAE-ს approximირებს და საშუალებას იძლევა ღრმა (max_depth 10–20, ასევე unconstrained + `lossguide`/`max_leaves`) კონფიგურაციები უსაფრთხოდ გამოვცადოთ.
- **GPU ავტოდეტექცია** (`device='cuda'` თუ ხელმისაწვდომია) სწრაფი, ღრმა search-ისთვის.
- MLflow/DagsHub/Drive კავშირები და run structure (`XGBoost_EDA` → `XGBoost_Cleaning` → `XGBoost_Feature_Selection` → `XGBoost_CrossValidation` → `XGBoost_HyperparameterTuning` → `XGBoost_Final`) უცვლელია.

**მნიშვნელოვანი შენიშვნა:** hold-out `val_wmae` არის დროზე დაფუძნებული split (train-ის ბოლო 10%) შედეგი — ეს საშუალებას აძლევს lag-დაფუძნებულ ფიჩერებს (`lag_1` და მისნაირები) ძალიან ძლიერი სიგნალი მისცენ მოდელს, რადგან იგივე Store×Dept სერიები რამდენიმე კვირით ადრეც გვხვდება train-ში. ეს მეტრიკა კარგად აჩვენებს pipeline-ის ხარისხს ამ split-ზე, თუმცა პირდაპირ არ არის შედარებადი ოფიციალურ Kaggle private leaderboard-თან (სადაც test დროში სრულიად მომავალშია და გამარჯვებულმა გუნდმაც კი 2348 WMAE მიაღწია — ensemble-ით, არა ცალკე XGBoost-ით).

**შემდეგი:** N-BEATS, ARIMA/SARIMA, PatchTST notebook-ები.